In [ ]:
from __future__ import annotations
import logging
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import cv2

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm
from tqdm import tqdm

from IPython.display import display

# ===================== Logging configuration =====================
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
print('Reading data with assigned folds')
display(pd.read_csv('/kaggle/input/0-create-fold/folds_by_Species.csv').head())
# ===================== Configuration class =====================
class CFG:
    # Paths
    base_path = Path("/kaggle/input/csiro-biomass")
    train_csv = "/kaggle/input/0-create-fold/folds_by_random.csv"
    train_image_dir = base_path / "train"
    output_dir = Path("./models_eva02_single")

    # Model and image settings
    model_name = "eva02_base_patch14_448.mim_in22k_ft_in1k"  # timm model name
    img_size = 448
    dropout = 0.1
    num_targets = 5  # Predict all 5 targets directly

    # Training hyperparameters
    batch_size = 8
    num_workers = 2
    num_epochs = 20
    lr = 1e-4
    weight_decay = 1e-5

    # Random seed
    seed = 42
    # Device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # Target column names, in wide-table order
    target_cols = ["Dry_Green_g","Dry_Dead_g","Dry_Clover_g","GDM_g","Dry_Total_g",]
    loss_weights = [0.1, 0.1, 0.1, 0.2, 0.5]  # In the same order as target_cols
    wandb_project = 'CSIRO_mac14'
CFG.MODEL_ = CFG.model_name.split('.')[0]    
# Create the output directory
CFG.output_dir.mkdir(parents=True, exist_ok=True)

# Set the random seed
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG.seed)

In [ ]:
# ===================== Data augmentation interface =====================
def get_train_transforms(img_size):
    """Training data augmentation (customize as needed)."""
    return A.Compose(
        [
            # Random horizontal flip
            A.HorizontalFlip(p=0.5),
            # Random vertical flip
            A.VerticalFlip(p=0.2),
            # Resize to a common size
            A.Resize(img_size, img_size),
            # Normalize
            A.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
            ToTensorV2(),
        ]
    )


def get_valid_transforms(img_size):
    """Validation preprocessing without random augmentation."""
    return A.Compose(
        [
            A.Resize(img_size, img_size),
            A.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
            ToTensorV2(),
        ]
    )

In [ ]:
import torch
import torch.nn as nn

class WeightedMSELoss(nn.Module):
    """Column-weighted MSE for multi-target regression with the 5 targets in this task."""
    def __init__(self, weights):
        super().__init__()
        # weights: list or 1D tensor of length num_targets
        # Register as a buffer so it moves to the corresponding device automatically
        self.register_buffer("weights", torch.tensor(weights).view(1, -1))

    def forward(self, preds, targets):
        """
        preds, targets: [batch_size, num_targets], in log space.
        """
        # Compute elementwise squared errors: [B, T]
        diff2 = (preds - targets) ** 2

        # Weight each column by its corresponding weight
        # weights has shape [1, T] and broadcasts to [B, T]
        weighted_diff2 = diff2 * self.weights

        # Two common averaging conventions are:
        # 1) Mean over all elements, equivalent to sum / (B * T)
        loss = weighted_diff2.mean()

        # To normalize by the total weight instead, use the following expression:
        # loss = weighted_diff2.sum() / (preds.size(0) * self.weights.sum())

        return loss

class TrainBiomassDataset(Dataset):
    """
    Training dataset:
    - Input: a single image stream.
    - Output: 5 log1p-transformed target values.
    """
    def __init__(self, df, image_dir, target_cols, transform):
        """
        df: wide table with one image_path and multiple target columns per row.
        image_dir: directory containing the images.
        target_cols: list of target column names.
        transform: Albumentations augmentation.
        """
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.image_paths = self.df["image_path"].values
        self.target_cols = target_cols
        self.transform = transform

        # Extract the original target values from df as float32
        targets = self.df[self.target_cols].values.astype(np.float32)

        # Apply log1p (log(1+y)) to the targets to stabilize training
        self.targets_log = np.log1p(targets)

    def __len__(self):
        """Return the dataset size."""
        return len(self.df)

    def __getitem__(self, idx):
        """Return one sample: image tensor and log1p-transformed target tensor."""

        # Get the image path
        img_path = self.image_paths[idx]

        # Join only the filename to the train directory, matching inference
        full_path = self.image_dir / Path(img_path).name

        # Read the image with cv2 (BGR by default)
        image = cv2.imread(str(full_path))

        # If reading fails, issue a warning and substitute an all-zero image
        if image is None:
            logger.warning(f"Failed to load image: {full_path}, using zeros.")
            image = np.zeros((1000, 2000, 3), dtype=np.uint8)

        # BGR -> RGB to match the model input format
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Apply Albumentations augmentation
        augmented = self.transform(image=image)
        image_tensor = augmented["image"]  # tensor，shape: (3, H, W)

        # Get the log1p-transformed target
        target_log = self.targets_log[idx]
        target_tensor = torch.from_numpy(target_log)

        return image_tensor, target_tensor
class SimpleBiomassModel(nn.Module):
    """
    Single-stream image regression model:
    - Extract image features with a timm backbone.
    - Use a fully connected regression head for 5 predictions in log space.
    """
    def __init__(self, model_name, num_targets, dropout=0.3):
        super().__init__()

        # Create a timm backbone without its classification head
        self.backbone = timm.create_model(
            model_name,
            pretrained=True,  # Pretrained weights can help convergence
            num_classes=0,    # Omit the classification head and use feature outputs
            global_pool="avg",  # Use average pooling to obtain a global vector
        )

        # Backbone output feature dimension
        n_features = self.backbone.num_features

        # Regression head: a two-layer MLP
        self.head = nn.Sequential(
            nn.Linear(n_features, n_features // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(n_features // 2, num_targets),
        )

    def forward(self, x):
        """
        Forward pass:
        x: input image tensor [B, C, H, W].
        Returns: predictions in log space [B, num_targets].
        """
        # Extract features
        feat = self.backbone(x)

        # Regression output in log space
        out = self.head(feat)

        return out


In [ ]:
# ===================== Single-epoch training with optional EMA =====================
def train_one_epoch(model, loader, optimizer, device, ema=None):
    """
    Train for one epoch and return mean weighted MSE loss in log space.

    Parameters:
    - model: model being trained.
    - loader: training DataLoader.
    - optimizer: optimizer.
    - device: execution device (CPU/GPU).
    - ema: optional ModelEMA instance, updated after each step when supplied.
    """
    model.train()
    running_loss = 0.0

    # Use weighted MSE in log space with different weights for the 5 targets
    loss_weights = [0.1, 0.1, 0.1, 0.2, 0.5]  # In the same order as target_cols
    criterion = WeightedMSELoss(loss_weights).to(device)

    for images, targets_log in tqdm(loader, desc="Train", leave=False):
        # images: [B, C, H, W]
        # targets_log: [B, 5], already transformed externally with log1p or a similar transform
        images = images.to(device)
        targets_log = targets_log.to(device)

        optimizer.zero_grad()
        preds_log = model(images)          # [B, 5]

        # Calculate weighted MSE loss in log space
        loss = criterion(preds_log, targets_log)

        loss.backward()
        optimizer.step()

        # Update EMA parameters after each step when an EMA object is supplied
        if ema is not None:
            ema.update(model)

        # Accumulate batch loss, weighted by batch size
        running_loss += loss.item() * images.size(0)

    # Return the epoch loss averaged over all samples
    epoch_loss = running_loss / len(loader.dataset)
    return epoch_loss


# ===================== Validation function =====================
def validate(model, loader, device):
    """
    Validation loop returning:
    - avg_loss: weighted MSE in the original scale for custom monitoring.
    - weighted_r2: weighted R-squared in the original scale using the official formula.
    """
    model.eval()

    # Validation target weights, matching the competition
    weights = np.array([0.1, 0.1, 0.1, 0.2, 0.5])  # The order must match target_cols

    running_loss = 0.0
    all_true = []
    all_pred = []

    with torch.no_grad():
        for images, targets_log in tqdm(loader, desc="Valid", leave=False):
            images = images.to(device)
            targets_log = targets_log.to(device)

            # Model outputs remain in log space
            preds_log = model(images)

            # Invert the log transform: y = exp(log_y) - 1
            preds = torch.expm1(preds_log).cpu().numpy()
            trues = torch.expm1(targets_log).cpu().numpy()

            all_pred.append(preds)
            all_true.append(trues)

            # ---------- Weighted MSE in the original scale for monitoring -----------
            diff2 = (preds - trues) ** 2        # [B, 5]
            weighted_diff2 = diff2 * weights    # Broadcast to [B, 5]
            batch_loss = weighted_diff2.mean()  # Average over all elements
            running_loss += batch_loss * preds.shape[0]

    # Concatenate all samples
    y_true = np.concatenate(all_true, axis=0)   # [N, 5]
    y_pred = np.concatenate(all_pred, axis=0)   # [N, 5]

    # Average the loss over all samples
    avg_loss = running_loss / len(y_true)

    # ---------- Calculate weighted R-squared using the official formula -----------
    weighted_r2 = weighted_r2_score(y_true, y_pred, weights)

    return avg_loss, weighted_r2


# ===================== Weighted R-squared using the official definition =====================
def weighted_r2_score(y_true, y_pred, weights_per_target):
    """
    y_true, y_pred: numpy arrays of shape [N, T], with T=5.
    weights_per_target: a 1D array of length T, e.g. [0.1, 0.1, 0.1, 0.2, 0.5].
    Calculate global weighted R-squared using the official formula.

    Mathematical form:
        R_w^2 = 1 - sum_j w_j (y_j - ŷ_j)^2 / sum_j w_j (y_j - ȳ_w)^2
        where ȳ_w = sum_j w_j y_j / sum_j w_j
    """
    # Convert to floating-point numpy arrays
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    w_t = np.asarray(weights_per_target, dtype=np.float64)   # [T]

    # Flatten all (sample, target) pairs into one dimension
    N, T = y_true.shape
    y_true_flat = y_true.reshape(-1)   # [N*T]
    y_pred_flat = y_pred.reshape(-1)   # [N*T]

    # Generate the corresponding weight w_j for each element
    # Repeat the [T] weights N times and flatten to [N*T]
    w_flat = np.broadcast_to(w_t, (N, T)).reshape(-1)

    # Global weighted mean ȳ_w
    y_bar_w = np.sum(w_flat * y_true_flat) / (np.sum(w_flat) + 1e-8)

    # Weighted residual sum of squares SS_res
    ss_res = np.sum(w_flat * (y_true_flat - y_pred_flat) ** 2)

    # Weighted total sum of squares SS_tot
    ss_tot = np.sum(w_flat * (y_true_flat - y_bar_w) ** 2)

    # Weighted R-squared
    r2_w = 1.0 - ss_res / (ss_tot + 1e-8)
    return float(r2_w)


# ===================== Custom EMA for arbitrary models =====================
from copy import deepcopy

class ModelEMA:
    """
    Simple exponential moving average (EMA) for any PyTorch model.

    Example:
        ema = ModelEMA(model, decay=0.999)
        ...
        # After each step
        ema.update(model)
        ...
        # Use ema.ema_model for validation
        valid_loss, valid_r2 = validate(ema.ema_model, valid_loader, device)
    """
    def __init__(self, model, decay=0.999):
        # Deep-copy the model; the EMA model does not update gradients
        self.ema_model = deepcopy(model)
        self.ema_model.eval()

        # EMA decay: values closer to 1 produce smoother updates
        self.decay = decay

        # Disable gradient computation for EMA parameters
        for p in self.ema_model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        """EMA update: ema = decay * ema + (1 - decay) * model."""
        ema_state = self.ema_model.state_dict()
        model_state = model.state_dict()

        for key in ema_state.keys():
            # ema_param = decay * ema_param + (1-decay) * new_param
            ema_state[key].mul_(self.decay).add_(model_state[key], alpha=1 - self.decay)


In [ ]:
import wandb
import gc
from kaggle_secrets import UserSecretsClient

# ================================
# Five-fold training with wandb and simple early stopping
# ================================
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("wandb")
wandb.login(key=secret_value_0)

for csv_path in [
    '/kaggle/input/0-create-fold/folds_by_random.csv',
    '/kaggle/input/0-create-fold/folds_by_Pre_GSHH_NDVI.csv',
    '/kaggle/input/0-create-fold/folds_by_Sampling_Date.csv',
    '/kaggle/input/0-create-fold/folds_by_Species.csv',
]:
    CFG.train_csv = csv_path
    Group_mode = csv_path.split('/')[-1].split('.')[0].split('_')[-1]

    train_wide = pd.read_csv(CFG.train_csv)
    fold_scores = []

    for fold_idx in range(5):

        # ============================
        # Initialize wandb
        # ============================
        run = wandb.init(
            project=CFG.wandb_project,
            name=f"{Group_mode}_{CFG.MODEL_}_fold{fold_idx}_size{CFG.img_size}_bs{CFG.batch_size}_lr{CFG.lr}",
            config={
                "Group_mode": Group_mode,
                "model_name": CFG.model_name,
                "fold_idx": fold_idx,
                "image_size": int(CFG.img_size),
                "batch_size": int(CFG.batch_size),
                "lr": float(CFG.lr),
                "num_epochs": int(CFG.num_epochs),
            },
            reinit=True,
        )

        # ============================
        # Split training and validation data
        # ============================
        df_train = train_wide[train_wide["fold"] != fold_idx].reset_index(drop=True)
        df_valid = train_wide[train_wide["fold"] == fold_idx].reset_index(drop=True)

        train_transform = get_train_transforms(CFG.img_size)
        valid_transform = get_valid_transforms(CFG.img_size)

        train_dataset = TrainBiomassDataset(
            df=df_train,
            image_dir=CFG.train_image_dir,
            target_cols=CFG.target_cols,
            transform=train_transform,
        )
        valid_dataset = TrainBiomassDataset(
            df=df_valid,
            image_dir=CFG.train_image_dir,
            target_cols=CFG.target_cols,
            transform=valid_transform,
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=CFG.batch_size,
            shuffle=True,
            num_workers=CFG.num_workers,
            pin_memory=True,
        )
        valid_loader = DataLoader(
            valid_dataset,
            batch_size=CFG.batch_size,
            shuffle=False,
            num_workers=CFG.num_workers,
            pin_memory=True,
        )

        # ============================
        # Model and optimizer
        # ============================
        model = SimpleBiomassModel(
            model_name=CFG.model_name,
            num_targets=len(CFG.target_cols),
            dropout=CFG.dropout,
        ).to(CFG.device)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=CFG.lr,
            weight_decay=CFG.weight_decay,
        )

        # ============================
        # Early Stopping
        # ============================
        patience = 5
        es_counter = 0
        best_r2 = -1e9
        best_epoch = 0
        best_model_path = CFG.output_dir / f"{Group_mode}_best_model_fold{fold_idx}.pth"

        # ============================
        # Basic training loop
        # ============================
        for epoch in range(1, CFG.num_epochs + 1):

            logger.info(f"[{Group_mode}] Fold {fold_idx} Epoch {epoch}/{CFG.num_epochs}")

            # Train
            train_loss = train_one_epoch(model, train_loader, optimizer, CFG.device)

            # Validate
            valid_loss, valid_r2 = validate(model, valid_loader, CFG.device)

            print(
                f"[{Group_mode}] Fold {fold_idx} | Epoch {epoch} | "
                f"Train Loss: {train_loss:.4f} | Valid Loss: {valid_loss:.4f} | "
                f"Valid R2: {valid_r2:.4f}"
            )

            # Log to wandb
            wandb.log(
                {
                    "epoch": epoch,
                    "fold": fold_idx,
                    "Group_mode": Group_mode,
                    "train_loss": train_loss,
                    "valid_loss": valid_loss,
                    "valid_r2": valid_r2,
                }
            )

            # ============================
            # Early stopping logic
            # ============================
            if valid_r2 > best_r2:
                best_r2 = valid_r2
                best_epoch = epoch
                es_counter = 0
                torch.save(model.state_dict(), best_model_path)
                wandb.run.summary["best_valid_r2"] = best_r2
                wandb.run.summary["best_epoch"] = best_epoch
                print(f"  ✓ New best model saved: {best_model_path}")
            else:
                es_counter += 1
                print(f"  ✗ No improvement ({es_counter}/{patience})")

            if es_counter >= patience:
                print(f"  🔥 Early stopping triggered at epoch {epoch}")
                break

        print(f"[{Group_mode}] Fold {fold_idx} Best R2: {best_r2:.4f} at epoch {best_epoch}")
        fold_scores.append(best_r2)

        wandb.finish()

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ============================
    # Summarize
    # ============================
    print(f"\n========== 5-FOLD RESULTS ({Group_mode}) ==========")
    for i, score in enumerate(fold_scores):
        print(f"Fold {i} R2: {score:.4f}")

    print(f"\n⭐ Average R2 ({Group_mode}): {np.mean(fold_scores):.4f}\n")
